<!-- chinese-cell-note -->
#### Cell 01 中文说明

**用途**：配置敏感性分析的路径、随机种子、绘图样式和输出目录，为 VIF、SRC、Bootstrap 与 SHAP 结果提供统一边界。

**输入与输出**：输入为 Step 1 数据位置；输出为 outputs_step2 和图件目录。

**评委回应重点**：它保证统计分析结果可定位、可复查、可复现。


<!-- english-cell-note -->
#### Cell 01 - Environment and sensitivity-analysis outputs

**Purpose**: This cell configures paths, seeds, plotting style, and output folders for the sensitivity-analysis workflow.

**Inputs and outputs**: Defines deterministic paths, folder handles, plotting defaults, and configuration objects used by downstream cells.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linestyle": "--",
})

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "outputs_step2"
FIG_DIR = OUT_DIR / "figures"
for p in [OUT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "step1_simulation_dataset.csv"
TARGET = "eui_kwh_m2"   # Start with EUI.
RANDOM_SEED = 42

<!-- chinese-cell-note -->
#### Cell 02 中文说明

**用途**：读取 Step 1 数据，并生成方向编码、几何派生量和窗型哑变量，确保 SRC、SHAP 和 ML 使用同一特征语义。

**输入与输出**：输入为 step1_simulation_dataset.csv；输出为分析矩阵、目标变量和特征分组。

**评委回应重点**：它避免不同 notebook 之间出现特征定义漂移。


<!-- english-cell-note -->
#### Cell 02 - Data loading and feature engineering

**Purpose**: This cell loads Step 1 data and creates orientation, geometry, and window-type features shared across later analyses.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 1) Load data ----------
assert DATASET_PATH.exists(), "Please run Step 1 first to generate step1_simulation_dataset.csv"
df = pd.read_csv(DATASET_PATH)
assert TARGET in df.columns, f"Target column missing from the dataset: {TARGET}"

# 1) Encode orientation as circular features.
df["orientation_sin"] = np.sin(np.deg2rad(df["orientation_deg"]))
df["orientation_cos"] = np.cos(np.deg2rad(df["orientation_deg"]))
df["footprint_area_m2"] = df["building_length"] * df["building_width"]
df["aspect_ratio"] = df["building_length"] / df["building_width"]

# 2) Convert window type into dummy variables.
df = pd.get_dummies(df, columns=["window_type_id"], prefix="window_type", drop_first=True)

# ---------- 1b) Geometry-based validation metrics ----------
df["building_height_calc_m"] = df["floor_num"] * df["floor_height"]

# Recover length and width from footprint area and aspect ratio.
L = np.sqrt(df["footprint_area_m2"] * df["aspect_ratio"])
W = np.sqrt(df["footprint_area_m2"] / df["aspect_ratio"])
P = 2 * (L + W)

# Envelope area divided by gross floor area (approximate).
df["envelope_to_floor_ratio"] = (
    P * df["building_height_calc_m"] + 2 * df["footprint_area_m2"]
) / df["gross_floor_area_m2"]

# 3) Final features used in this analysis step.
analysis_features = [
    'insul_thick', 'wwr', 'wall_thick',
    'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
    'u_wall', 'u_roof', 'u_ground',
    'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
    'roof_insul_thick',

    # Building geometry after reconstruction.
    'floor_num', 'footprint_area_m2', 'aspect_ratio', 'floor_height',
    'orientation_sin', 'orientation_cos',

    # Internal function and thermal-zone variables.
    'public_area', 'room_area', 'room_count',

    # Operation and system variables.
    'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
    'cool_set', 'heat_set', 'dhw_temp',
    'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
    'fresh_air_ach', 'operation_hours',

    # Window-type dummy variables.
    'window_type_2', 'window_type_3'
]

feature_groups = {
    "Envelope": [
        'insul_thick', 'wwr', 'wall_thick',
        'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
        'u_wall', 'u_roof', 'u_ground',
        'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
        'roof_insul_thick', 'window_type_2', 'window_type_3'
    ],
    "Geometry_Form": [
        'floor_num', 'footprint_area_m2', 'aspect_ratio', 'floor_height',
        'orientation_sin', 'orientation_cos'
    ],
    "Program_Zoning": [
        'public_area', 'room_area', 'room_count'
    ],
    "Operation_HVAC": [
        'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
        'cool_set', 'heat_set', 'dhw_temp',
        'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
        'fresh_air_ach', 'operation_hours'
    ]
}

feature_fullname_map = {
    "insul_thick": "External wall insulation layer thickness (m)",
    "wwr": "Overall window-to-wall ratio (-)",
    "wall_thick": "External wall structure thickness (m)",

    "u_win_n": "Heat transfer coefficient of north-facing windows (W/(m²·K))",
    "u_win_s": "Heat transfer coefficient of south-facing windows (W/(m²·K))",
    "u_win_e": "Heat transfer coefficient of east-facing windows (W/(m²·K))",
    "u_win_w": "Heat transfer coefficient of west-facing windows (W/(m²·K))",

    "u_wall": "Heat transfer coefficient of main external wall structure (W/(m²·K))",
    "u_roof": "Heat transfer coefficient of roof (W/(m²·K))",
    "u_ground": "Heat transfer coefficient of ground foundation (W/(m²·K))",

    "shgc_n": "Solar heat gain coefficient of north-facing windows (-)",
    "shgc_s": "Solar heat gain coefficient of south-facing windows (-)",
    "shgc_e": "Solar heat gain coefficient of east-facing windows (-)",
    "shgc_w": "Solar heat gain coefficient of west-facing windows (-)",

    "roof_insul_thick": "Roof insulation layer thickness (m)",

    "floor_num": "Total number of floors (floors)",
    "footprint_area_m2": "Building footprint area = main building length × main building width (m²)",
    "aspect_ratio": "Building length-to-width ratio (-)",
    "floor_height": "Standard floor height (m)",

    "orientation_sin": "Sine term of overall building orientation (-)",
    "orientation_cos": "Cosine term of overall building orientation (-)",

    "public_area": "Total area of public areas (m²)",
    "room_area": "Floor area of a single guest room (m²)",
    "room_count": "Total number of guest rooms (rooms)",

    "equip_power": "Indoor equipment power density (W/m²)",
    "dhw_per_person": "Daily domestic hot water consumption per capita (m³/(person·d))",
    "occupancy_density": "Personnel density (person/m²)",
    "light_power": "Indoor lighting power density (W/m²)",

    "cool_set": "Summer cooling set temperature (°C)",
    "heat_set": "Winter heating set temperature (°C)",
    "dhw_temp": "Domestic hot water supply temperature (°C)",

    "cop_cooling": "Rated COP of refrigeration unit (-)",
    "cop_heating": "COP of heat pump heating (-)",
    "boiler_eff": "Thermal efficiency of hot water boiler (-)",
    "fan_eff": "Efficiency of air conditioning fan (-)",
    "fresh_air_ach": "Fresh air exchange rate (times/h)",
    "operation_hours": "Total annual operating hours of system (h)",

    "window_type_2": "Window construction type dummy: Type 2 relative to Type 1 baseline (-)",
    "window_type_3": "Window construction type dummy: Type 3 relative to Type 1 baseline (-)",
}

X = df[analysis_features].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature count:", len(analysis_features))

<!-- chinese-cell-note -->
#### Cell 03 中文说明

**用途**：计算 VIF 以识别几何和围护结构变量之间的共线性，为解释 SRC 系数提供边界条件。

**输入与输出**：输入为候选特征矩阵；输出为 vif_table.csv 和高 VIF 排序。

**评委回应重点**：它回应线性系数稳定性和可解释性方面的审稿疑问。


<!-- english-cell-note -->
#### Cell 03 - Multicollinearity diagnosis

**Purpose**: This cell computes VIF values to diagnose multicollinearity before interpreting SRC magnitudes.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 2) VIF check ----------
def compute_vif_table(X_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in X_df.columns:
        y_col = X_df[col].values
        X_other = X_df.drop(columns=[col]).values
        model = LinearRegression()
        model.fit(X_other, y_col)
        r2 = model.score(X_other, y_col)
        vif = np.inf if r2 >= 0.999999 else 1.0 / (1.0 - r2)
        rows.append({"feature": col, "VIF": vif})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False)

vif_df = compute_vif_table(X)
vif_df["feature_full"] = vif_df["feature"].map(feature_fullname_map)
vif_df = vif_df[["feature", "feature_full", "VIF"]]

vif_df.to_csv(OUT_DIR / "vif_table.csv", index=False, encoding="utf-8-sig")
display(vif_df)

<!-- chinese-cell-note -->
#### Cell 04 中文说明

**用途**：定义带 Bootstrap 置信区间和符号稳定性判断的 SRC 函数，供 EUI、site energy 和 OCEI 对比重复使用。

**输入与输出**：输入为数据表、特征列表、目标变量和重采样次数；输出为 SRC、CI 和稳定性标签。

**评委回应重点**：它使敏感性指标从单次回归结果升级为可审计统计过程。


<!-- english-cell-note -->
#### Cell 04 - Reusable SRC estimator

**Purpose**: This cell defines a reusable SRC estimator with bootstrap confidence intervals and sign-stability assessment.

**Inputs and outputs**: Computes SRC estimates, uncertainty summaries, and ranked sensitivity evidence for manuscript reporting.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
from tools.sensitivity_utils import run_src_model

print("Shared robust SRC utility loaded from tools/sensitivity_utils.py")

<!-- chinese-cell-note -->
#### Cell 05 中文说明

**用途**：用交叉验证评估线性模型对 EUI 响应面的解释能力，明确 SRC 的适用范围。

**输入与输出**：输入为特征矩阵与 eui_kwh_m2；输出为 CV R2、RMSE 和预测残差信息。

**评委回应重点**：它承认并量化 SRC 的线性假设边界。


<!-- english-cell-note -->
#### Cell 05 - Linear-approximation diagnostic

**Purpose**: This cell evaluates how well a linear approximation represents the EUI response surface through cross-validation.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 3) Check usability of the linear approximation ----------
def cv_rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
X_std = StandardScaler().fit_transform(X)

pred_cv = cross_val_predict(
    LinearRegression(),
    X_std,
    y,
    cv=cv
)

print({
    "cv_r2": round(r2_score(y, pred_cv), 4),
    "cv_rmse": round(cv_rmse(y, pred_cv), 4),
})

<!-- chinese-cell-note -->
#### Cell 06 中文说明

**用途**：在全样本 SRC 基础上进行 Bootstrap 重采样，估计不确定性并标注符号是否稳定。

**输入与输出**：输入为特征矩阵和 EUI；输出为 src_indices_bootstrap.csv。

**评委回应重点**：它避免仅凭单次回归排序选择关键变量。


<!-- english-cell-note -->
#### Cell 06 - Bootstrap SRC estimation

**Purpose**: This cell estimates SRC values with bootstrap uncertainty intervals and sign-stability flags.

**Inputs and outputs**: Computes SRC estimates, uncertainty summaries, and ranked sensitivity evidence for manuscript reporting.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 4) SRC with bootstrap resampling ----------
def fit_src(X_df: pd.DataFrame, y_series: pd.Series):
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_std = x_scaler.fit_transform(X_df)
    y_std = y_scaler.fit_transform(y_series.to_numpy().reshape(-1, 1)).ravel()

    model = LinearRegression()
    model.fit(X_std, y_std)
    return model.coef_

# SRC fitted on all samples.
coef_full = fit_src(X, y)

# Bootstrap
B = 1000
rng = np.random.default_rng(RANDOM_SEED)
coef_boot = np.zeros((B, X.shape[1]))

for b in range(B):
    idx = rng.integers(0, len(X), len(X))
    Xb = X.iloc[idx].reset_index(drop=True)
    yb = y.iloc[idx].reset_index(drop=True)
    coef_boot[b, :] = fit_src(Xb, yb)

src_df = pd.DataFrame({
    "feature": X.columns,
    "SRC": coef_full,
    "abs_SRC": np.abs(coef_full),
    "CI_low": np.quantile(coef_boot, 0.025, axis=0),
    "CI_high": np.quantile(coef_boot, 0.975, axis=0),
})

src_df["sign_stable"] = (
    (src_df["CI_low"] > 0) | (src_df["CI_high"] < 0)
)

src_df = src_df.sort_values("abs_SRC", ascending=False).reset_index(drop=True)
src_df["feature_full"] = src_df["feature"].map(feature_fullname_map)

src_df = src_df[
    ["feature", "feature_full", "SRC", "abs_SRC", "CI_low", "CI_high", "sign_stable"]
]

src_df.to_csv(OUT_DIR / "src_indices_bootstrap.csv", index=False, encoding="utf-8-sig")
display(src_df.head(20))

<!-- chinese-cell-note -->
#### Cell 07 中文说明

**用途**：使用 XGBoost SHAP 值补充 SRC 的线性视角，检验非线性模型下的重要性排序是否一致。

**输入与输出**：输入为同一组分析变量和 EUI；输出为 SHAP 排序、SRC-SHAP 对比表和解释图。

**评委回应重点**：它直接回应非线性响应面下 SRC 是否充分的问题。


<!-- english-cell-note -->
#### Cell 07 - SHAP-based nonlinear cross-check

**Purpose**: This cell uses XGBoost SHAP values as a nonlinear cross-check of the SRC ranking.

**Inputs and outputs**: Builds the nonlinear SHAP cross-check and exports rank-comparison tables and figures.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ============================================================
# [IMPROVEMENT P0-4 & P1-7] Test-set SHAP sensitivity analysis
#
# Uses held-out SHAP values and hyperparameter-robustness checks so the
# nonlinear cross-check is not tied to the training data or to one XGBoost setting.
# ============================================================

import warnings
try:
    from tqdm import TqdmWarning
    warnings.filterwarnings("ignore", message="IProgress not found.*", category=TqdmWarning)
except Exception:
    warnings.filterwarnings("ignore", message="IProgress not found.*")

import shap
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

X_train_shap, X_test_shap, y_train_shap, y_test_shap = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, shuffle=True
)

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=1,
    verbosity=0,
)
xgb.fit(X_train_shap, y_train_shap)

explainer = shap.TreeExplainer(xgb)
shap_values_train = explainer.shap_values(X_train_shap)
shap_values_test = explainer.shap_values(X_test_shap)

shap_train_importance = pd.DataFrame({
    "feature": X.columns,
    "shap_abs_mean_train": np.abs(shap_values_train).mean(axis=0),
})
shap_test_importance = pd.DataFrame({
    "feature": X.columns,
    "shap_abs_mean": np.abs(shap_values_test).mean(axis=0),
})
shap_importance = shap_test_importance.sort_values("shap_abs_mean", ascending=False).reset_index(drop=True)

shap_stability = shap_train_importance.merge(shap_test_importance, on="feature")
shap_stability["rank_train"] = shap_stability["shap_abs_mean_train"].rank(ascending=False, method="first")
shap_stability["rank_test"] = shap_stability["shap_abs_mean"].rank(ascending=False, method="first")
shap_stability["abs_rank_shift"] = (shap_stability["rank_train"] - shap_stability["rank_test"]).abs()
train_test_r, train_test_p = spearmanr(shap_stability["rank_train"], shap_stability["rank_test"])

xgb_alt = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_SEED + 7,
    n_jobs=1,
    verbosity=0,
)
xgb_alt.fit(X_train_shap, y_train_shap)
alt_explainer = shap.TreeExplainer(xgb_alt)
shap_values_alt_test = alt_explainer.shap_values(X_test_shap)
alt_importance = pd.DataFrame({
    "feature": X.columns,
    "shap_abs_mean_alt": np.abs(shap_values_alt_test).mean(axis=0),
})
shap_hyperparameter_check = shap_test_importance.merge(alt_importance, on="feature")
shap_hyperparameter_check["rank_primary"] = shap_hyperparameter_check["shap_abs_mean"].rank(ascending=False, method="first")
shap_hyperparameter_check["rank_alternative"] = shap_hyperparameter_check["shap_abs_mean_alt"].rank(ascending=False, method="first")
hyper_r, hyper_p = spearmanr(shap_hyperparameter_check["rank_primary"], shap_hyperparameter_check["rank_alternative"])

compare_df = src_df[["feature", "abs_SRC", "SRC"]].merge(shap_importance, on="feature", how="left")
compare_df["src_rank"] = compare_df["abs_SRC"].rank(ascending=False, method="first")
compare_df["shap_rank"] = compare_df["shap_abs_mean"].rank(ascending=False, method="first")
compare_df["rank_delta"] = compare_df["src_rank"] - compare_df["shap_rank"]
compare_df["src_norm"] = compare_df["abs_SRC"] / compare_df["abs_SRC"].max()
compare_df["shap_norm"] = compare_df["shap_abs_mean"] / compare_df["shap_abs_mean"].max()
compare_df["importance_disagreement"] = (compare_df["src_norm"] - compare_df["shap_norm"]).abs()

src_shap_r = compare_df["src_rank"].corr(compare_df["shap_rank"], method="spearman")
src_top18 = set(compare_df.nsmallest(18, "src_rank")["feature"])
shap_top18 = set(compare_df.nsmallest(18, "shap_rank")["feature"])
overlap = len(src_top18 & shap_top18)
jaccard = overlap / len(src_top18 | shap_top18)

print("=" * 60)
print("SRC vs TEST-SET SHAP VARIABLE RANKING COMPARISON")
print("=" * 60)
print(f"SHAP train-test rank Spearman r: {train_test_r:.4f} (p={train_test_p:.4e})")
print(f"SHAP primary-vs-alternative XGBoost rank Spearman r: {hyper_r:.4f} (p={hyper_p:.4e})")
print(f"SRC vs test-set SHAP rank Spearman r: {src_shap_r:.4f}")
print(f"SRC top-18 / SHAP top-18 overlap: {overlap}/18 (Jaccard={jaccard:.3f})")
print(f"Top 10 variables by SRC:  {compare_df.nsmallest(10, 'src_rank')['feature'].tolist()}")
print(f"Top 10 variables by SHAP: {compare_df.nsmallest(10, 'shap_rank')['feature'].tolist()}")

print("\nSRC-SHAP DIVERGENCE ANALYSIS")
divergent = compare_df.reindex(compare_df["rank_delta"].abs().sort_values(ascending=False).index).head(10)
for _, row in divergent.iterrows():
    if row["rank_delta"] < 0:
        direction = "SHAP ranks it higher, suggesting nonlinear or interaction effects"
    else:
        direction = "SRC ranks it higher, suggesting a stronger linear main effect"
    print(f"  {row['feature']:<25} | SRC rank={int(row['src_rank']):>2}, SHAP rank={int(row['shap_rank']):>2}, delta={int(row['rank_delta']):>+3} | {direction}")

fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=150)
top20_src = compare_df.nsmallest(20, "src_rank").iloc[::-1]
axes[0].barh(top20_src["feature"], top20_src["abs_SRC"], color="#4C72B0")
axes[0].set_xlabel("|SRC|", fontsize=12)
axes[0].set_title("Top 20 Variables by SRC", fontsize=13)
axes[0].grid(axis="x", alpha=0.3)

top20_shap = compare_df.nsmallest(20, "shap_rank").iloc[::-1]
bar_colors = ["#4C72B0" if f in src_top18 else "darkorange" for f in top20_shap["feature"]]
axes[1].barh(top20_shap["feature"], top20_shap["shap_abs_mean"], color=bar_colors)
axes[1].set_xlabel("Mean |SHAP Value| on Held-Out Test Set", fontsize=12)
axes[1].set_title("Top 20 Variables by Test-Set SHAP", fontsize=13)
axes[1].grid(axis="x", alpha=0.3)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#4C72B0", label="In both SRC and SHAP top-18"), Patch(facecolor="darkorange", label="SHAP-only (not in SRC top-18)")]
axes[1].legend(handles=legend_elements, fontsize=9, loc="lower right")
plt.tight_layout()
out_dir = PROJECT_ROOT / "outputs_step2" / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "src_vs_shap_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 8), dpi=150)
shap.summary_plot(shap_values_test, X_test_shap, show=False, max_display=20)
plt.tight_layout()
fig2.savefig(out_dir / "shap_summary_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

compare_df.to_csv(PROJECT_ROOT / "outputs_step2" / "src_shap_ranking_comparison.csv", index=False, encoding="utf-8-sig")
shap_stability.sort_values("abs_rank_shift", ascending=False).to_csv(PROJECT_ROOT / "outputs_step2" / "shap_train_test_rank_stability.csv", index=False, encoding="utf-8-sig")
shap_hyperparameter_check.sort_values("rank_primary").to_csv(PROJECT_ROOT / "outputs_step2" / "shap_hyperparameter_robustness.csv", index=False, encoding="utf-8-sig")
compare_df.sort_values("importance_disagreement", ascending=False).to_csv(PROJECT_ROOT / "outputs_step2" / "src_shap_divergence_analysis.csv", index=False, encoding="utf-8-sig")
print("\nSHAP robustness and SRC-SHAP comparison tables saved.")


<!-- chinese-cell-note -->
#### Cell 08 中文说明

**用途**：通过 |SRC| 碎石图、累计贡献和交叉验证曲线论证 18 变量截断，而不是使用任意阈值。

**输入与输出**：输入为 SRC 排序和候选特征；输出为变量截断图与 cv_r2_by_variable_count.csv。

**评委回应重点**：它为变量 15-18 是否应保留提供定量证据。


<!-- english-cell-note -->
#### Cell 08 - Variable-cutoff justification

**Purpose**: This cell justifies the 18-variable cutoff using SRC scree structure, cumulative contribution, and CV performance.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Justifies retained variables with quantitative stability evidence rather than a visual or arbitrary cutoff.


In [ ]:
# ============================================================
# [IMPROVEMENT P1-7] Multi-criterion variable-selection cutoff analysis
#
# Tests whether the 18-variable cutoff is supported by SRC magnitude,
# cumulative contribution, and predictive-power curves across model classes.
# ============================================================

src_sorted = src_df.sort_values("abs_SRC", ascending=False).reset_index(drop=True)
src_sorted["cumulative_abs_src"] = src_sorted["abs_SRC"].cumsum()
src_sorted["cumulative_pct"] = src_sorted["cumulative_abs_src"] / src_sorted["abs_SRC"].sum() * 100
src_sorted["incremental_src"] = src_sorted["abs_SRC"].diff().abs()
src_sorted["pct_of_max"] = src_sorted["abs_SRC"] / src_sorted["abs_SRC"].max() * 100

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
models_for_cutoff = {
    "LinearRegression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LinearRegression())]),
    "RidgeCV": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=5))]),
    "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=200, max_depth=6, random_state=RANDOM_SEED, n_jobs=1))]),
}

cv_rows = []
for model_name, model in models_for_cutoff.items():
    for n in range(1, len(X.columns) + 1):
        top_n_features = src_sorted.head(n)["feature"].tolist()
        scores = cross_val_score(model, X[top_n_features], y, cv=cv, scoring="r2", n_jobs=1)
        cv_rows.append({"model": model_name, "n_vars": n, "cv_r2_mean": scores.mean(), "cv_r2_std": scores.std(ddof=1)})
cv_df = pd.DataFrame(cv_rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)
ax = axes[0]
colors = ["#C44E52" if i < 18 else "#4C72B0" for i in range(len(src_sorted))]
ax.bar(range(1, len(src_sorted) + 1), src_sorted["abs_SRC"], color=colors, width=0.7)
ax.axvline(18.5, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
ax.text(19, ax.get_ylim()[1] * 0.95, "Cutoff at 18 variables", fontsize=9, ha="left")
ax.set_xlabel("Variable Rank (by |SRC|)", fontsize=12)
ax.set_ylabel("|SRC|", fontsize=12)
ax.set_title("SRC Magnitude Scree Plot", fontsize=13)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
ax.plot(range(1, len(src_sorted) + 1), src_sorted["cumulative_pct"], "o-", color="#C44E52", markersize=4, linewidth=1.5)
ax.axhline(90, color="grey", linestyle=":", alpha=0.5, label="90% cumulative |SRC|")
ax.axhline(95, color="grey", linestyle="--", alpha=0.5, label="95% cumulative |SRC|")
ax.axvline(18, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
ax.fill_between([1, 18], 0, 100, alpha=0.05, color="green")
ax.text(10, 50, f"First 18 variables:\n{src_sorted.iloc[17]['cumulative_pct']:.1f}% of total |SRC|", fontsize=9, ha="center")
ax.set_xlabel("Number of Variables Included", fontsize=12)
ax.set_ylabel("Cumulative |SRC| (%)", fontsize=12)
ax.set_title("Cumulative SRC Contribution", fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[2]
palette = {"LinearRegression": "#4C72B0", "RidgeCV": "darkorange", "RandomForest": "seagreen"}
for model_name, model_df in cv_df.groupby("model"):
    ax.plot(model_df["n_vars"], model_df["cv_r2_mean"], "o-", markersize=3, linewidth=1.3, color=palette[model_name], label=model_name)
    peak = model_df.loc[model_df["cv_r2_mean"].idxmax()]
    ax.axvline(peak["n_vars"], color=palette[model_name], linestyle=":", alpha=0.25)
ax.axvline(18, color="black", linestyle="--", linewidth=1.5, alpha=0.7, label="Selected cutoff (18)")
ax.set_xlabel("Number of Variables Included", fontsize=12)
ax.set_ylabel("5-Fold CV R?", fontsize=12)
ax.set_title("Predictive Power vs Variable Count", fontsize=13)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs_step2" / "figures" / "variable_selection_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

cutoff_summary = []
for model_name, model_df in cv_df.groupby("model"):
    peak = model_df.loc[model_df["cv_r2_mean"].idxmax()]
    score_18 = model_df.loc[model_df["n_vars"] == 18, "cv_r2_mean"].iloc[0]
    cutoff_summary.append({"model": model_name, "peak_n_vars": int(peak["n_vars"]), "peak_cv_r2": float(peak["cv_r2_mean"]), "cv_r2_at_18": float(score_18), "delta_peak_minus_18": float(peak["cv_r2_mean"] - score_18)})
cutoff_summary_df = pd.DataFrame(cutoff_summary)

print("\nVARIABLE SELECTION ANALYSIS")
print(f"Cumulative |SRC| at n=10: {src_sorted.iloc[9]['cumulative_pct']:.1f}%")
print(f"Cumulative |SRC| at n=18: {src_sorted.iloc[17]['cumulative_pct']:.1f}%")
print(f"Cumulative |SRC| at n=25: {src_sorted.iloc[24]['cumulative_pct']:.1f}%")
print("\nMulti-model cutoff summary:")
print(cutoff_summary_df.round(5).to_string(index=False))
cv_df.to_csv(PROJECT_ROOT / "outputs_step2" / "cv_r2_by_variable_count.csv", index=False, encoding="utf-8-sig")
cutoff_summary_df.to_csv(PROJECT_ROOT / "outputs_step2" / "variable_cutoff_multimodel_summary.csv", index=False, encoding="utf-8-sig")


<!-- chinese-cell-note -->
#### 变量排名边界稳定性说明

**用途**：量化 top-18 边界附近变量的置信区间宽度，说明 15-22 位弱影响变量的排序可能出现小幅波动。

**输入与输出**：输入为 SRC 排名表；输出为 `variable_ranking_stability.csv`。

**评委回应重点**：它避免把第 18 位解释成绝对分界，而是把变量筛选表述为主导变量稳定、边界变量透明报告。


<!-- english-cell-note -->
#### Variable-Ranking Boundary Stability

**Purpose**: This cell quantifies confidence-interval width around the top-18 boundary, where weak-effect variables may exchange ranks.

**Inputs and outputs**: It uses the SRC ranking table and exports `variable_ranking_stability.csv`.

**Reviewer-facing rationale**: The output supports a calibrated interpretation: dominant variables are stable, whereas boundary variables are reported transparently rather than overclaimed.


In [ ]:
# [P2-11] Stability of variable ranking near the top-18 boundary
stability_df = src_sorted.copy()
stability_df["ci_width"] = stability_df["CI_high"] - stability_df["CI_low"]
stability_df["rank"] = np.arange(1, len(stability_df) + 1)
stability_df["rank_stability_class"] = pd.cut(
    stability_df["abs_SRC"],
    bins=[-np.inf, 0.015, 0.05, np.inf],
    labels=["tail (<0.015)", "boundary (0.015-0.05)", "core (>0.05)"],
)
stability_df[[
    "rank", "feature", "SRC", "abs_SRC", "CI_low", "CI_high",
    "ci_width", "sign_stable", "rank_stability_class"
]].to_csv(OUT_DIR / "variable_ranking_stability.csv", index=False, encoding="utf-8-sig")

print("Variable-ranking stability table saved:", OUT_DIR / "variable_ranking_stability.csv")
display(stability_df.loc[stability_df["rank"].between(15, 22), [
    "rank", "feature", "abs_SRC", "ci_width", "sign_stable", "rank_stability_class"
]])

### SRC 方法的线性假设局限与补充验证

**针对审稿人关于 SRC 线性假设的回应：**

标准化回归系数（SRC）本质上基于普通最小二乘（OLS）回归，其有效性与模型的线性拟合优度密切相关。当响应面存在显著非线性时，SRC 可能低估通过交互作用或阈值效应产生影响的关键变量。

为评估 SRC 在当前数据集上的适用性并补充非线性视角，本研究增加了以下分析：

1. **SHAP 值排序验证**：使用 XGBoost（一种基于树的非线性模型）计算 SHAP（SHapley Additive exPlanations）值，与 SRC 排序进行交叉验证。Spearman 秩相关系数衡量两种方法排序的一致性。

2. **累积贡献分析**：绘制 SRC 绝对值从大到小的累积贡献曲线，识别自然断点，为 18 变量截断阈值提供定量依据。

3. **预测能力曲线**：通过 5 折交叉验证的线性回归 R² 随变量数量变化曲线，评估增加变量对预测能力的边际贡献，确定收益递减的拐点。


<!-- chinese-cell-note -->
#### Cell 09 中文说明

**用途**：将单变量 SRC 汇总为物理变量组，识别 DHW、运行、HVAC、围护结构和几何机制的相对贡献。

**输入与输出**：输入为 feature_groups 与 src_df；输出为 src_group_summary.csv。

**评委回应重点**：它把统计排序转化为可讨论的建筑物理机制。


<!-- english-cell-note -->
#### Cell 09 - Group-level sensitivity summary

**Purpose**: This cell aggregates feature-level SRC results into physical groups for engineering interpretation.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 5) Group-level summary ----------
group_rows = []
for gname, feats in feature_groups.items():
    sub = src_df[src_df["feature"].isin(feats)].copy()
    group_rows.append({
        "group": gname,
        "n_features": len(feats),
        "sum_abs_SRC": sub["abs_SRC"].sum(),
        "mean_abs_SRC": sub["abs_SRC"].mean(),
        "n_sign_stable": int(sub["sign_stable"].sum())
    })

group_df = pd.DataFrame(group_rows).sort_values("sum_abs_SRC", ascending=False)
group_df.to_csv(OUT_DIR / "src_group_summary.csv", index=False, encoding="utf-8-sig")
display(group_df)

<!-- chinese-cell-note -->
#### Cell 10 中文说明

**用途**：重新展示所有输入变量的 SRC 排名并标注前 18 个变量，使关键变量选择具有完整背景。

**输入与输出**：输入为分析特征和 EUI；输出为 src_current_all_inputs.csv 与全变量 SRC 图。

**评委回应重点**：它防止关键变量列表脱离完整参数空间。


<!-- english-cell-note -->
#### Cell 10 - All-input SRC ranking plot

**Purpose**: This cell visualizes SRC rankings for all input variables while marking the selected top 18.

**Inputs and outputs**: Reads the validated step outputs and writes English-only manuscript figures to the documented figures folder.

**Reviewer-facing rationale**: Translates the EUI-OCEI divergence into ranked evidence that can be checked against the manuscript claims.


In [ ]:
src_current = run_src_model(df, analysis_features, TARGET, seed=42, B=1000)
src_current["rank_abs"] = src_current["abs_SRC"].rank(method="first", ascending=False)
src_current["is_top18"] = src_current["rank_abs"] <= 18

src_current["feature_full"] = src_current["feature"].map(feature_fullname_map)

src_current.to_csv(OUT_DIR / "src_current_all_inputs.csv", index=False, encoding="utf-8-sig")

plot_df = src_current.sort_values("abs_SRC", ascending=True).copy()
colors = ["#C44E52" if x else "#4C72B0" for x in plot_df["is_top18"]]

fig, ax = plt.subplots(figsize=(12.5, 12.5))
ax.barh(plot_df["feature"], plot_df["abs_SRC"], color=colors)
ax.set_title("SRC of ALL input variables")
ax.set_xlabel("|SRC|")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(FIG_DIR / "src_all_current_inputs_top18_marked.png", dpi=300, bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 11 中文说明

**用途**：绘制前 18 个变量的有符号 SRC，说明关键变量对 EUI 的正向或负向边际影响。

**输入与输出**：输入为 src_df 前 18 行；输出为 src_top18.png。

**评委回应重点**：它把变量筛选结果转化为工程可解释结论。


<!-- english-cell-note -->
#### Cell 11 - Top-18 signed SRC figure

**Purpose**: This cell plots signed SRC values for the top 18 variables to show direction as well as magnitude.

**Inputs and outputs**: Reads the validated step outputs and writes English-only manuscript figures to the documented figures folder.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 6) Top SRC ----------
plot_df = src_df.head(18).iloc[::-1].copy()

fig, ax = plt.subplots(figsize=(8.8, 7.0))
ax.barh(plot_df["feature"], plot_df["SRC"])
ax.axvline(0, color="black", linewidth=1.0)
ax.set_title("Top 18 Standardized Regression Coefficients (SRC)")
ax.set_xlabel("SRC")
fig.tight_layout()
fig.savefig(FIG_DIR / "src_top18.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 12 中文说明

**用途**：用颜色区分 Bootstrap 中符号稳定与不稳定的关键变量，提示排序结果的不确定性。

**输入与输出**：输入为前 18 个 SRC 变量及 sign_stable；输出为稳定性条形图。

**评委回应重点**：它让审稿人看到变量重要性并非无不确定性的点估计。


<!-- english-cell-note -->
#### Cell 12 - Sign-stability visualization

**Purpose**: This cell visualizes sign stability for top SRC variables using bootstrap-derived markers.

**Inputs and outputs**: Reads the validated step outputs and writes English-only manuscript figures to the documented figures folder.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 7) Stability markers ----------
sel = src_df.head(18).copy().sort_values("abs_SRC", ascending=True)

fig, ax = plt.subplots(figsize=(8.8, 7.0))
colors = ["#C44E52" if s else "#4C72B0" for s in sel["sign_stable"]]
ax.barh(sel["feature"], sel["abs_SRC"], color=colors)
ax.set_title("Top 18 |SRC| (red = sign-stable, blue = sign-unstable)")
ax.set_xlabel("|SRC|")
fig.tight_layout()
fig.savefig(FIG_DIR / "src_abs_top18_stability.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 13 中文说明

**用途**：绘制几何相关变量与 EUI/site energy 的散点关系，直观检查几何约束和响应趋势。

**输入与输出**：输入为 envelope_to_floor_ratio、floor_num、EUI 和 site energy；输出为几何诊断图。

**评委回应重点**：它补充解释几何变量在 SRC 排名中的作用。


<!-- english-cell-note -->
#### Cell 13 - Geometry-response diagnostics

**Purpose**: This cell plots geometry-response relationships for visual diagnostics of EUI and site-energy behavior.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(df["envelope_to_floor_ratio"], df["eui_kwh_m2"], alpha=0.7)
axes[0].set_xlabel("envelope_to_floor_ratio")
axes[0].set_ylabel("eui_kwh_m2")
axes[0].set_title("Envelope-to-floor ratio vs EUI")

axes[1].scatter(df["floor_num"], df["eui_kwh_m2"], alpha=0.7)
axes[1].set_xlabel("floor_num")
axes[1].set_ylabel("eui_kwh_m2")
axes[1].set_title("Floor number vs EUI")

axes[2].scatter(df["floor_num"], df["site_energy_kwh"], alpha=0.7)
axes[2].set_xlabel("floor_num")
axes[2].set_ylabel("site_energy_kwh")
axes[2].set_title("Floor number vs Site Energy")

plt.tight_layout()
plt.show()

<!-- chinese-cell-note -->
#### Cell 14 中文说明

**用途**：分别对 EUI 与总场地能耗运行 SRC，比较几何变量在强度指标和总量指标中的作用差异。

**输入与输出**：输入为统一特征列表和两个目标变量；输出为对比表。

**评委回应重点**：它说明面积归一化指标与总能耗指标的解释逻辑不同。


<!-- english-cell-note -->
#### Cell 14 - EUI versus site-energy SRC comparison

**Purpose**: This cell compares SRC patterns for EUI and total site energy, focusing on geometry-related variables.

**Inputs and outputs**: Computes SRC estimates, uncertainty summaries, and ranked sensitivity evidence for manuscript reporting.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
src_eui = run_src_model(df, analysis_features, "eui_kwh_m2", seed=42, B=1000)
src_site = run_src_model(df, analysis_features, "site_energy_kwh", seed=42, B=1000)

compare_df = pd.DataFrame({
    "feature": analysis_features,
    "SRC_EUI": src_eui.set_index("feature").reindex(analysis_features)["SRC"].values,
    "SRC_site_energy": src_site.set_index("feature").reindex(analysis_features)["SRC"].values,
}).sort_values("SRC_EUI", key=np.abs, ascending=False)

display(compare_df.loc[
    compare_df["feature"].isin([
        "floor_num", "footprint_area_m2", "floor_height",
        "aspect_ratio", "public_area", "room_area", "room_count"
    ])
])
compare_df.to_csv(OUT_DIR / "src_target_comparison.csv", index=False, encoding="utf-8-sig")

<!-- chinese-cell-note -->
#### Cell 15 中文说明

**用途**：整理物理变量集合并导出最终变量选择依据，作为 Step 3 模型训练的接口。

**输入与输出**：输入为 SRC/SHAP 结果和候选物理变量；输出为关键变量文件与补充诊断。

**评委回应重点**：它闭合从敏感性筛选到代理模型训练的方法链路。


<!-- english-cell-note -->
#### Cell 15 - Physical-feature consolidation

**Purpose**: This cell consolidates the physics-informed feature list and exports selected-variable evidence for downstream modelling.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
physics_features = [
    'insul_thick', 'wwr', 'wall_thick',
    'u_win_n', 'u_win_s', 'u_win_e', 'u_win_w',
    'u_wall', 'u_roof', 'u_ground',
    'shgc_n', 'shgc_s', 'shgc_e', 'shgc_w',
    'roof_insul_thick',
    'envelope_to_floor_ratio',
    'public_area', 'room_area', 'room_count',
    'equip_power', 'dhw_per_person', 'occupancy_density', 'light_power',
    'cool_set', 'heat_set', 'dhw_temp',
    'cop_cooling', 'cop_heating', 'boiler_eff', 'fan_eff',
    'fresh_air_ach', 'operation_hours',
    'orientation_sin', 'orientation_cos',
    'window_type_2', 'window_type_3'
]

src_eui_physics = run_src_model(df, physics_features, "eui_kwh_m2", seed=42, B=1000)
display(src_eui_physics.head(15))

src_eui_physics.to_csv(
    OUT_DIR / "src_physical_validation.csv",
    index=False,
    encoding="utf-8-sig"
)